# Module 1 — Data Prep (FGADR + IDRiD)

Builds the classification pkl files DRG-Net needs and produces one real sample
visualization (image + combined lesion mask + grade + Lesion Burden Score) for the
presentation's data section (b.1).

**Manual prerequisite (one-time):** upload these 4 zips to a Google Drive folder, e.g.
`My Drive/Thesis_Datasets/`:
- `FGADR-Seg-set_Release.zip`
- `archive.zip` (IDRiD, from Kaggle)
- `FIRE_dataset.zip`
- `LongDRScreening_20150209.zip`

This notebook only needs FGADR + IDRiD; FIRE/LongDR are unzipped here too since notebook 04
needs them later and it's cheap to do once.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os
# This repo is PRIVATE -- Colab needs a GitHub Personal Access Token to clone it.
# Create one (once, reusable across all 4 notebooks/sessions) at
# https://github.com/settings/tokens -> "Generate new token (classic)" -> scope: repo.
# Input is hidden; not saved anywhere.
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'  # <-- change if you used a different folder
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR} -- upload the 4 dataset zips there first"


In [ ]:
# Unzip datasets locally on the Colab VM disk (much faster I/O than reading zips off Drive
# directly). -n skips files that already exist, so this is safe/cheap to re-run.
os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FGADR-Seg-set_Release.zip"
!unzip -q -n "$DRIVE_DATA_DIR/archive.zip" -d IDRiD_dataset_root
!unzip -q -n "$DRIVE_DATA_DIR/FIRE_dataset.zip"
!unzip -q -n "$DRIVE_DATA_DIR/LongDRScreening_20150209.zip"

# archive.zip may unzip with an extra nesting level; normalize so IDRiD_dataset ends up
# directly under /content/data
import glob, shutil
candidates = glob.glob('/content/data/IDRiD_dataset_root/**/IDRiD_dataset', recursive=True)
if candidates and not os.path.isdir('/content/data/IDRiD_dataset'):
    shutil.move(candidates[0], '/content/data/IDRiD_dataset')
print('IDRiD_dataset present:', os.path.isdir('/content/data/IDRiD_dataset'))


In [ ]:
# Clone this repo (private -- uses the token above) and the pinned DRG-Net reference
# implementation. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir('/content/M2-DRProgression'):
    !git clone --branch module1-fgadr-poc https://{GITHUB_TOKEN}@github.com/bearawr/M2-DRProgression.git M2-DRProgression
if not os.path.isdir('/content/dr-joint-learning'):
    !git clone https://github.com/DFKI-Interactive-Machine-Learning/dr-joint-learning.git dr-joint-learning
    %cd dr-joint-learning
    !git checkout 0da1bbe885f438390a0e94ec486283d9b21d5547
    %cd /content

# Compatibility patch: DRG-Net's own code (dr_classification/data/transforms.py) still uses
# torchvision.transforms.RandomAffine's old 'fillcolor' kwarg, renamed to 'fill' and fully
# removed in the torchvision version Colab now ships (we deliberately don't install the repo's
# old pinned torchvision -- see notebook 02's install cell -- so this old kwarg name breaks).
# Idempotent: sed -i only changes the file if 'fillcolor' is still present.
!sed -i 's/fillcolor=aug_args.value_fill/fill=aug_args.value_fill/' /content/dr-joint-learning/dr_classification/data/transforms.py

# Same class of issue, in dr_segmentation's transform code (only exercised by notebook 03,
# harmless to run here too): functional.py imports PILLOW_VERSION, removed from Pillow years
# ago; transforms_group.py bare-imports ipdb, a debugger package not installed by default.
# Both would crash the import chain immediately (from transform.transforms_group import *).
!pip install -q ipdb
_func_py = '/content/dr-joint-learning/dr_segmentation/transform/functional.py'
_txt = open(_func_py).read()
_txt = _txt.replace(
    'from PIL import Image, ImageOps, ImageEnhance, PILLOW_VERSION',
    "from PIL import Image, ImageOps, ImageEnhance\nPILLOW_VERSION = '9.0.0'"
)
open(_func_py, 'w').write(_txt)



In [ ]:
# Copy DRG-Net's own filtered-label CSV (vendored in our repo) into the local FGADR folder --
# dr_segmentation/utils.py::get_images_fgadr_from_pd reads this file from inside IMAGE_DIR.
# Made robust to zip-nesting variation (some zip tools add/drop the top-level
# FGADR-Seg-set_Release wrapper folder) by locating Seg-set/ wherever it actually landed.
import glob as _glob, os, shutil

_candidates = _glob.glob('/content/data/**/Seg-set', recursive=True)
assert _candidates, (
    'No Seg-set folder found under /content/data. Run: !ls /content/data  and  '
    '!ls "$DRIVE_DATA_DIR"  to check the zip is named exactly FGADR-Seg-set_Release.zip '
    'and actually unzipped in the previous cell.'
)
fgadr_segset_dir = _candidates[0]
print('Found FGADR Seg-set at:', fgadr_segset_dir)

expected = '/content/data/FGADR-Seg-set_Release/Seg-set'
if fgadr_segset_dir != expected:
    os.makedirs(os.path.dirname(expected), exist_ok=True)
    if not os.path.exists(expected):
        os.symlink(fgadr_segset_dir, expected)
    print(f'Normalized path: {expected} -> {fgadr_segset_dir}')

shutil.copy(
    '/content/M2-DRProgression/module1/data/DR_Seg_Grading_Label_Filtered.csv',
    os.path.join(expected, 'DR_Seg_Grading_Label_Filtered.csv')
)
print('done')


In [ ]:
# Build the classification pkl files (grade classifier pretraining data)
!pip install -q pandas
import os
os.makedirs('/content/drive/MyDrive/Thesis_Datasets/module1_pkl', exist_ok=True)
%cd /content/M2-DRProgression
!python module1/dataprep/make_fgadr_classification_pkl.py \
    --fgadr-root /content/data/FGADR-Seg-set_Release/Seg-set \
    --out /content/drive/MyDrive/Thesis_Datasets/module1_pkl/fgadr_classification_pkl.pkl
!python module1/dataprep/make_idrid_classification_pkl.py \
    --idrid-root "/content/data/IDRiD_dataset/B. Disease Grading" \
    --out /content/drive/MyDrive/Thesis_Datasets/module1_pkl/idrid_classification_pkl.pkl


In [ ]:
# Real sample: image + combined lesion mask + grade + Lesion Burden Score (b.1 deliverable)
%cd /content/M2-DRProgression/module1
!python compute_lbs.py --fgadr-root /content/data/FGADR-Seg-set_Release/Seg-set \
    --image-name 0003_3.png --out /content/module1_sample.png
from IPython.display import Image as IPImage, display
display(IPImage('/content/module1_sample.png'))


## Data shapes (for the presentation)

- FGADR: 1,842 images, 5-class ICDR grade label, 6 lesion mask types (we use MA/HE/EX/SE per
  the manuscript's scope); 401 of 1,842 have all 4 in common (DRG-Net's own filtered subset).
- IDRiD grading set: 413 train / 103 test images, same 5-class grade label.
- Images: variable native resolution (FGADR/IDRiD originals), resized to 512x512 for both the
  classifier and segmentation models (matching DRG-Net's own configs).
- Lesion Burden Score (LBS) = lesion pixels / retinal FOV pixels -- see the sample image above.
